# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for col in ["impressions_90d", "days_since_last_update", "word_count", "avg_position", "ctr"]:
    print(f"\n{col}:")
    print(df[col].describe().round(2))


impressions_90d:
count     30000.00
mean       5200.37
std       16838.02
min           1.00
25%          81.00
50%         731.00
75%        3615.25
max      517715.00
Name: impressions_90d, dtype: float64

days_since_last_update:
count    30000.00
mean        46.10
std         42.08
min          1.00
25%         20.00
50%         20.00
75%        104.00
max        373.00
Name: days_since_last_update, dtype: float64

word_count:
count    22301.00
mean      3107.76
std       1452.38
min          8.00
25%       2413.00
50%       2877.00
75%       3666.00
max       9546.00
Name: word_count, dtype: float64

avg_position:
count    30000.00
mean        16.34
std         15.22
min          0.00
25%          6.20
50%         10.80
75%         22.30
max        245.00
Name: avg_position, dtype: float64

ctr:
count    30000.00
mean         0.51
std          3.28
min          0.00
25%          0.00
50%          0.07
75%          0.29
max        100.00
Name: ctr, dtype: float64


Heavy tails observed: impressions_90d has mean 5,200 but median only 731 — a small number of very high-traffic pages pull the mean far above typical. ctr shows the same pattern (mean 0.51, median 0.07, max 100) — likely a few outlier/data-entry values, since CTR should not exceed 1.0. word_count has ~26% missing values (22,301 of 30,000 rows). These heavy tails mean any model using raw impressions_90d or ctr should consider log-transforms or capping, and any average-based summary should be read alongside the median, not instead of it.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: word count vs decline
df["wc_bucket"] = pd.cut(df["word_count"], bins=[0,1000,2000,3000,4000,10000], labels=["<1k","1k-2k","2k-3k","3k-4k","4k+"])
t1 = df.groupby("wc_bucket", observed=True).agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).round(3)
print("Signal 1 — word_count vs decline:")
print(t1)

# Signal 2: content age vs decline
df["age_bucket"] = pd.cut(df["content_age_days"], bins=[0,90,180,365,730,10000], labels=["<90d","90-180d","180-365d","1-2yr","2yr+"])
t2 = df.groupby("age_bucket", observed=True).agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).round(3)
print("\nSignal 2 — content_age vs decline:")
print(t2)

# Signal 3: position vs decline
df["pos_bucket"] = pd.cut(df["avg_position"], bins=[0,3,10,20,50,1000], labels=["1-3","4-10","11-20","21-50","50+"])
t3 = df.groupby("pos_bucket", observed=True).agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).round(3)
print("\nSignal 3 — avg_position vs decline:")
print(t3)

Signal 1 — word_count vs decline:
              n  decline_rate
wc_bucket                    
<1k         973         0.207
1k-2k      3781         0.555
2k-3k      7985         0.587
3k-4k      5198         0.588
4k+        4364         0.604

Signal 2 — content_age vs decline:
                n  decline_rate
age_bucket                     
<90d          492         0.669
90-180d     11780         0.626
180-365d    11368         0.515
1-2yr        6360         0.426

Signal 3 — avg_position vs decline:
                n  decline_rate
pos_bucket                     
1-3          1141         0.498
4-10        11842         0.569
11-20        7273         0.610
21-50        7225         0.562
50+          1314         0.343


Signal 1 — word_count vs decline → verdict: OPPOSITE (of the common belief "longer content = better/safer"). Decline rate is LOWEST for the shortest pages (<1k words, 0.207) and rises steadily with length, peaking at 0.604 for 4k+ words. This directly echoes Discovery C from notebook 01 — length is not protective; longer pages here are not safer from decline.

Signal 2 — content_age vs decline → verdict: CONFIRMED (older pages decline less, once past the newest bracket). Decline rate drops steadily from 0.669 (<90 days) down to 0.426 (1-2 years) — meaning genuinely new content actually declines MOST often in this sample, likely reflecting early volatility before a page's traffic stabilizes, while older, more established pages settle into fewer declines.

Signal 3 — avg_position vs decline → verdict: MIXED. Decline rate is not monotonic — it's 0.498 at top positions (1-3), rises to a peak of 0.610 at 11-20, then drops back down to 0.343 for 50+. This is not a clean "worse position = worse decline" story; pages ranked very poorly (50+) actually decline least often here, possibly because they have so little traffic to lose in the first place.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Staleness — behind FlyRank's refresh flags
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=[0,30,90,180,365,10000], labels=["<30d","30-90d","90-180d","180-365d","365d+"])
t4 = df.groupby("stale_bucket", observed=True).agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).round(3)
print("Flag-linked signal — staleness vs decline (behind refresh flags):")
print(t4)

Flag-linked signal — staleness vs decline (behind refresh flags):
                  n  decline_rate
stale_bucket                     
<30d          20480         0.511
30-90d          175         0.589
90-180d        9171         0.611
180-365d        169         0.467
365d+             5         0.600


Verdict: MIXED. Decline rate rises from 0.511 (<30d) to 0.611 (90-180d), which supports the refresh flag's core assumption over that range — but then drops to 0.467 at 180-365d, and the 365d+ bucket has only 5 rows, too few to trust. The data does NOT fully support "staler always means more likely to decline" as a standalone rule; it only holds up to about 180 days, after which the pattern breaks down. This matches the same finding from my Week-4 baseline audit (ML-07).

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*
A content team should not treat word count, position, or staleness alone as reliable single-signal triggers for review — each one only tells part of the story, and two (word count, staleness) show patterns that are the OPPOSITE or WEAKER than commonly assumed. The clearest, most trustworthy pattern found here is content age: very new pages decline most often, likely due to early instability rather than quality issues, and this should temper any urge to "fix" brand-new pages too aggressively. Combining signals (as the Week-4 baseline rule already does, using staleness AND visibility together) is more defensible than relying on any single metric shown here.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.